In [3]:
# ==========================================
# SANSKRIT-TO-ENGLISH NMT PIPELINE
# Assignment 2: Natural Language Understanding
# Name: Sakshi Verma | Roll Number: G25AIT1147
# ==========================================

# --- CELL 1: DEPENDENCY INSTALLATION ---
# Installs all mandated libraries including Hugging Face transformers, metrics tracking, and optimization tools
!pip install -q transformers[torch] datasets tokenizers bert-score nltk accelerate -U

# --- CELL 2: CORE IMPORTS & REPRODUCIBILITY SEEDING ---
import os
import time
import torch
import pandas as pd
import numpy as np
import nltk
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score as bert_score_fn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# Ensure NLTK tokenizer models are downloaded quietly
nltk.download('punkt', quiet=True)

# Set deterministic random seed vectors for metric consistency
torch.manual_seed(42)
np.random.seed(42)

# Hardware execution router checking for CUDA acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"System Check complete. Routing execution space to: {device}")


# --- CELL 3: DATA PREPROCESSING & DATA LOADER ---
def load_parallel_data(sa_path, en_path):
    """Loads and aligns Sanskrit and English parallel text files."""
    if os.path.exists(sa_path) and os.path.exists(en_path):
        df_sa = pd.read_csv(sa_path)
        df_en = pd.read_csv(en_path)
        merged = pd.merge(df_sa, df_en, on="Source_id")
        return merged
    else:
        print(f"Parallel files '{sa_path}' or '{en_path}' not found in runtime tree. Launching dummy dataset fallback.")
        # Fallback mock data structure
        mock_sa = pd.DataFrame({
            "Source_id": [1, 2],
            "Sentence_sa": ["Ctrl, S नुत्वा रक्षन्तु।", "गुरुः छात्रान् एकवारं पाठयति ।"]
        })
        mock_en = pd.DataFrame({
            "Source_id": [1, 2],
            "Sentence_en": ["Save it with Ctrl, S.", "Teacher will teach the students only once."]
        })
        return pd.merge(mock_sa, mock_en, on="Source_id")


# --- CELL 4: HIGH-SPEED PARALLEL INFERENCE & SUBMISSION ENGINE ---
def run_fast_submission_pipeline(test_sa_path="test_sa_1000.csv", output_path="submission.csv", batch_size=32):
    """
    Optimized Parallel Inference Pipeline:
    Batches texts to speed up calculations, handles cross-lingual mBART token structures,
    and exports a strictly compliant, unindexed UTF-8 encoded submission file.
    """
    if not os.path.exists(test_sa_path):
        print(f"Warning: The file '{test_sa_path}' was not found in this folder. Please verify the exact filename.")
        return

    print(f"\nInitializing Parallel Translation Engine | Batch Size: {batch_size}")

    # 1. Load data assets
    test_df = pd.read_csv(test_sa_path)
    source_ids = test_df["Source_id"].tolist()
    sanskrit_sentences = test_df["Sentence_sa"].tolist()
    predicted_english = []

    # 2. Setup the architecture checkpoints
    MODEL_NAME = "facebook/mbart-large-50"
    print(f"Loading weights from model base: {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.src_lang = "sa_IN"   # Sanskrit subword target locale
    tokenizer.tgt_lang = "en_XX"   # English generation locale

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    # Calculate architecture scale metrics
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Architecture Parameters Loaded: {total_params:,}")

    # 3. High-Speed Mini-Batch Parallel Execution Loop
    start_time = time.time()
    with torch.no_grad():
        for i in tqdm(range(0, len(sanskrit_sentences), batch_size), desc="Processing Translation Batches"):
            batch_texts = sanskrit_sentences[i : i + batch_size]

            # Contextually tokenize the entire batch simultaneously with padding vectors
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            ).to(device)

            # Multi-hypothesis generation using Beam Search
            generated_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

            # Decode the entire output batch together
            batch_outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            predicted_english.extend(batch_outputs)

    end_time = time.time()
    print(f"\nInference execution complete! Total processing time: {end_time - start_time:.2f} seconds.")

    # 4. Formulate submission structure exactly as required by the rubric
    submission_df = pd.DataFrame({
        "Source_id": source_ids,
        "Sentence_en": predicted_english
    })

    # Enforce strict compliance: No line index numbers, formatted explicitly with UTF-8
    submission_df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Success! Your compliant submission file has been exported to: {output_path}")

    # 5. Compute Required Validation Metrics (If ground truth targets are present)
    if "Sentence_en" in test_df.columns:
        print("\nCalculating required validation scores against ground truth maps...")
        references = test_df["Sentence_en"].tolist()

        # Calculate unweighted NLTK BLEU score
        bleu_scores = []
        for ref, pred in zip(references, predicted_english):
            ref_tokens = [nltk.word_tokenize(ref.lower())]
            pred_tokens = nltk.word_tokenize(pred.lower())
            score = sentence_bleu(ref_tokens, pred_tokens) # Uniform unweighted token baseline
            bleu_scores.append(score)

        # Calculate BERTScore with mandated baseline rescaling
        _, _, F1 = bert_score_fn(predicted_english, references, lang="en", rescale_with_baseline=True)

        print("\n================ SYSTEM METRICS ================ ")
        print(f"Mean NLTK BLEU Metric Score: {np.mean(bleu_scores):.4f}")
        print(f"Mean BERTScore F1 (Baseline Rescaled): {F1.mean().item():.4f}")
        print("================================================ ")

# --- CELL 5: RUN THE PIPELINE ---
# Change "test_sa_1000.csv" to your exact Sanskrit file path if it has a different name
run_fast_submission_pipeline("test_sa_1000.csv", batch_size=32)

System Check complete. Routing execution space to: cpu

Initializing Parallel Translation Engine | Batch Size: 32
Loading weights from model base: facebook/mbart-large-50...


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Architecture Parameters Loaded: 610,879,488



Processing Translation Batches: 100%|██████████| 32/32 [4:25:54<00:00, 498.59s/it]



Inference execution complete! Total processing time: 15954.99 seconds.
Success! Your compliant submission file has been exported to: submission.csv
